In [3]:
import re
import numpy as np
import pandas as pd
import requests, sys

ENSEMBL VEP configurations


In [4]:
server = "https://rest.ensembl.org"
ext = "/vep/human/hgvs"
headers = {"Content-Type": "application/json", "Accept": "application/json"}

## Import Beagle sgRNA annotation


In [5]:
variants = pd.read_csv(f"../data/BeagleCoelho/EG_collab-guides.txt", sep="\t")
variants.shape

(24752, 24)

In [6]:
variants.sample(25)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,...,PAM Sequence,sgRNA Sequence Start Pos. (global),sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note
8995,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,AG,55202513,antisense,55202527G>A,C_6,3173G>A,Cys1058Tyr,Missense,NaN,NaN
19005,ENST00000621592.8,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000008.11,ENSG00000136997,MYC,+,...,CG,127738696,antisense,"127738709G>A, 127738710G>A","C_7, C_6","492G>A, 493G>A","Leu164Leu, Ala165Thr","Silent, Missense",NaN,NaN
5276,ENST00000275493.7,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,TG,55143474,antisense,55143490T>C,A_4,424+2T>C,(NC),Splice-donor,NaN,NaN
10605,ENST00000307102.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000015.10,ENSG00000169032,MAP2K1,+,...,GG,66485089,sense,66485092C>T;66485093C>T,C_4;C_5,796C>T;797C>T,Pro266Leu,Missense,NaN,NaN
16459,ENST00000429687.8,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000014.9,ENSG00000129484,PARP2,+,...,CG,20343632,antisense,"20343644G>A, 20343645G>A;20343647G>A, 20343648G>A","C_8, C_7;C_5, C_4","3G>A, 4G>A;6G>A, 7G>A","Met1Ile, Ala2Thr, Ala3Thr","Missense, Missense, Missense",NaN,NaN
16339,ENST00000366794.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,CG,226407834,antisense,"226407840C>T, 226407839C>T","C_7, C_6","90G>A, 91G>A","Lys30Lys, Asp31Asn","Silent, Missense",NaN,NaN
7196,ENST00000275493.7,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,TG,55165430,sense,55165434A>G,A_5,1877A>G,Tyr626Cys,Missense,NaN,NaN
12893,ENST00000366794.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,TG,226363126,antisense,226363131C>T,C_6,2816G>A,Ser939Asn,Missense,NaN,NaN
16536,ENST00000429687.8,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000014.9,ENSG00000129484,PARP2,+,...,CG,20343684,antisense,20343697T>C,A_7,46+10T>C,(NC),Intron,NaN,NaN
11664,ENST00000333681.5,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000018.10,ENSG00000171791,BCL2,-,...,GG,63128668,sense,63128681T>C,A_7,664A>G,Ser222Gly,Missense,NaN,NaN


In [7]:
variants.loc[7343]

Input                                                    ENST00000275493.7
CRISPR Enzyme                                                   SpyoCas9NG
Edit Type                                                              C-T
Edit Window                                                           4..8
Target Taxon                                                          9606
Target Assembly                                                     GRCh38
Target Genome Sequence                                        NC_000007.14
Target Gene ID                                             ENSG00000146648
Target Gene Symbol                                                    EGFR
Target Gene Strand                                                       +
Target Transcript ID                                     ENST00000275493.7
Target Domain                                                          CDS
sgRNA Sequence                                        ATCGCCACTGGGATGGTGGG
sgRNA Context Sequence   

## Format for ENSEMBL VEP input

Use the Beagle columns [input, Nucleotide Edits] and format as HGVS identifiers for ENSEMBL VEP. Examples include:

- `ENST00000618231.3:c.9G>C`
- `ENST00000471631.1:c.28_33delTCGCGG`
- `ENST00000285667.3:c.1047_1048insC`
- `5:g.140532G>C`


In [8]:
def beagle2vep(r: pd.Series) -> list[str]:
    """
    Create a list of ENSEMBL VEP HGVS notations from a row of the Beagle output.

    Args:
    r: pd.Series: A row of the Beagle output. Must contain the columns "Nucleotide Edits" and "Target Transcript ID".

    Returns:
    list[str]: A list of ENSEMBL VEP HGVS notations.

    """

    r_edits = r["Nucleotide Edits"].split(", ")

    return [f"{r['Target Transcript ID']}:c.{v}" for v in r_edits]

In [9]:
beagle2vep(variants.loc[7343])

['ENST00000275493.7:c.1940C>T;1941C>T', 'ENST00000275493.7:c.1943C>T']

In [125]:
variants_hgvs = [
    v
    for _, r in variants.iterrows()
    if r["Nucleotide Edits"] is not np.nan
    for v in beagle2vep(r)
]
np.random.choice(variants_hgvs, 25)

array(['ENST00000429687.8:c.199G>A', 'ENST00000366794.10:c.1212C>T',
       'ENST00000646891.2:c.745G>A', 'ENST00000621592.8:c.381C>T',
       'ENST00000621592.8:c.31-3C>T', 'ENST00000649815.2:c.1014C>T',
       'ENST00000649815.2:c.1153A>G', 'ENST00000646891.2:c.2065A>G',
       'ENST00000275493.7:c.454A>G;455A>G', 'ENST00000649815.2:c.222C>T',
       'ENST00000366794.10:c.2278-2A>G', 'ENST00000307102.10:c.135G>A',
       'ENST00000646891.2:c.227C>T', 'ENST00000275493.7:c.1006+3A>G',
       'ENST00000262948.10:c.1006G>A', 'ENST00000649815.2:c.616A>G',
       'ENST00000366794.10:c.74G>A', 'ENST00000307102.10:c.708G>A',
       'ENST00000429687.8:c.1553+4A>G',
       'ENST00000262948.10:c.187A>G;188A>G',
       'ENST00000275493.7:c.89-7C>T', 'ENST00000646891.2:c.173A>G',
       'ENST00000621592.8:c.437A>G', 'ENST00000366794.10:c.616G>A;617G>A',
       'ENST00000333681.5:c.506T>C'], dtype='<U44')

Write variants HGVS input to file


In [134]:
len(variants_hgvs)

26104

In [132]:
with open("../data/BeagleCoelho/EG_collab-guides-VEP-HGVS-input.txt", "w") as f:
    for variant in variants_hgvs:
        f.write(f"{variant}\n")

In [133]:
!head -n 10 ../data/BeagleCoelho/EG_collab-guides-VEP-HGVS-input.txt

ENST00000262948.10:c.*12C>T
ENST00000262948.10:c.*14C>T
ENST00000262948.10:c.*7C>T
ENST00000262948.10:c.*8C>T
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.1203A>G
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.*1C>T
ENST00000262948.10:c.*9G>A
ENST00000262948.10:c.*10G>A


VEP command


In [ ]:
# ./vep --af --af_1kg --af_gnomade --af_gnomadg --appris --biotype --buffer_size 500 --ccds --check_existing --distance 5000 --domains --gencode_primary --hgvs --mane --numbers --plugin OpenTargets,file=[path_to]/OTGenetics.tsv.gz --plugin CADD,snv=[path_to]/CADD_GRCh38_1.7_whole_genome_SNVs.tsv.gz,indels=[path_to]/CADD_GRCh38_1.7_InDels.tsv.gz --plugin REVEL,file=[path_to]/new_tabbed_revel_grch38.tsv.gz --plugin SpliceAI,snv=[path_to]/spliceai_scores.masked.snv.hg38.vcf.gz,indel=[path_to]/spliceai_scores.masked.indel.hg38.vcf.gz,snv_ensembl=[path_to]/spliceai_scores.raw.snv.ensembl_mane.grch38.110.vcf.gz --plugin Enformer,file=[path_to]/enformer_grch38.vcf.gz --plugin AlphaMissense,file=[path_to]/AlphaMissense_hg38.tsv.gz --plugin Blosum62 --plugin MaveDB,file=[path_to]/MaveDB_variants.tsv.gz --plugin ClinPred,file=[path_to]/ClinPred_hg38_sorted_tabbed.tsv.gz --plugin dbscSNV,[path_to]/dbscSNV1.1_GRCh38.txt.gz --plugin EVE,file=[path_to]/eve_merged.vcf.gz --plugin LOEUF,file=[path_to]/loeuf_dataset_grch38.tsv.gz,match_by=gene --plugin NMD --plugin AncestralAllele,[path_to]/homo_sapiens_ancestor_GRCh38_109.fa.gz --plugin MaxEntScan,[path_to]/maxentscan --plugin mutfunc,db=[path_to]/mutfunc_data.db,motif=1,int=1,mod=1,exp=1 --polyphen b --protein --pubmed --regulatory --show_ref_allele --sift b --species homo_sapiens --symbol --transcript_version --tsl --uniprot --uploaded_allele --var_synonyms --cache --input_file [input_data] --output_file [output_file]

In [1]:
!head -n 10 ../data/BeagleCoelho/EG_collab-guides-VEP-HGVS-output.txt

#Uploaded_variation	Location	Allele	Consequence	IMPACT	SYMBOL	Gene	Feature_type	Feature	BIOTYPE	EXON	INTRON	HGVSc	HGVSp	cDNA_position	CDS_position	Protein_position	Amino_acids	Codons	Existing_variation	REF_ALLELE	UPLOADED_ALLELE	DISTANCE	STRAND	FLAGS	SYMBOL_SOURCE	HGNC_ID	MANE	MANE_SELECT	MANE_PLUS_CLINICAL	TSL	APPRIS	CCDS	ENSP	SWISSPROT	TREMBL	UNIPARC	UNIPROT_ISOFORM	SIFT	PolyPhen	DOMAINS	HGVS_OFFSET	AF	AFR_AF	AMR_AF	EAS_AF	EUR_AF	SAS_AF	gnomADe_AF	gnomADe_AFR_AF	gnomADe_AMR_AF	gnomADe_ASJ_AF	gnomADe_EAS_AF	gnomADe_FIN_AF	gnomADe_MID_AF	gnomADe_NFE_AF	gnomADe_REMAINING_AF	gnomADe_SAS_AF	gnomADg_AF	gnomADg_AFR_AF	gnomADg_AMI_AF	gnomADg_AMR_AF	gnomADg_ASJ_AF	gnomADg_EAS_AF	gnomADg_FIN_AF	gnomADg_MID_AF	gnomADg_NFE_AF	gnomADg_REMAINING_AF	gnomADg_SAS_AF	CLIN_SIG	SOMATIC	PHENO	PUBMED	VAR_SYNONYMS	MOTIF_NAME	MOTIF_POS	HIGH_INF_POS	MOTIF_SCORE_CHANGE	TRANSCRIPTION_FACTORS	OpenTargets_geneId	OpenTargets_l2g	CADD_PHRED	CADD_RAW	REVEL	SpliceAI_pred_DP_AG	SpliceAI_pred_DP_AL	SpliceAI_pred_DP_DG

## Merge with data


## [Optional] Rest API to query a single edit


In [128]:
vep_request = {"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}

In [129]:
# dict to string
vep_request = str(vep_request).replace("'", '"')
vep_request

'{"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}'

In [130]:
r = requests.post(
    server + ext,
    headers=headers,
    data=vep_request,
)

if not r.ok:
    r.raise_for_status()
    sys.exit()

decoded = r.json()
print(repr(decoded))

[{'strand': 1, 'seq_region_name': '14', 'end': 20357524, 'transcript_consequences': [{'transcript_id': 'ENST00000250416', 'gene_id': 'ENSG00000129484', 'gene_symbol': 'PARP2', 'variant_allele': 'G', 'hgnc_id': 'HGNC:272', 'gene_symbol_source': 'HGNC', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'biotype': 'protein_coding', 'strand': 1, 'impact': 'LOW'}, {'hgnc_id': 'HGNC:272', 'gene_symbol_source': 'HGNC', 'variant_allele': 'G', 'impact': 'LOW', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'strand': 1, 'biotype': 'protein_coding', 'gene_id': 'ENSG00000129484', 'transcript_id': 'ENST00000429687', 'gene_symbol': 'PARP2'}, {'gene_id': 'ENSG00000129484', 'transcript_id': 'ENST00000527384', 'gene_symbol': 'PARP2', 'distance': 1569, 'hgnc_id': 'HGNC:272', 'gene_symbol_source': 'HGNC', 'variant_allele': 'G', 'impact': 'MODIFIER', 'consequence_terms': ['downstream_gene_variant'], 'biotype': 'retained_intron', 'strand': 1}, {'transcript_id'

In [131]:
variant_vep = decoded[0]
variant_vep

{'strand': 1,
 'seq_region_name': '14',
 'end': 20357524,
 'transcript_consequences': [{'transcript_id': 'ENST00000250416',
   'gene_id': 'ENSG00000129484',
   'gene_symbol': 'PARP2',
   'variant_allele': 'G',
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'biotype': 'protein_coding',
   'strand': 1,
   'impact': 'LOW'},
  {'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'LOW',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'strand': 1,
   'biotype': 'protein_coding',
   'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000429687',
   'gene_symbol': 'PARP2'},
  {'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000527384',
   'gene_symbol': 'PARP2',
   'distance': 1569,
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'MODIFIER',
   'consequence_terms': ['dow